<a href="https://colab.research.google.com/github/EngenhariaSoftwarePUCRS/Inteligencia_Artificial/blob/develop/Trabalho01/Trabalho01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [73]:
integrantes = ["Augusto Baldino", "Felipe Freitas", "Isabela Kuser", "Luiza Heller", "Maria Eduarda Maia", "Paola Lopes"]
useful_links = [
    "https://pandas.pydata.org/pandas-docs/version/1.0/user_guide/style.html",
    "https://www.geeksforgeeks.org/how-to-replace-values-in-column-based-on-condition-in-pandas/",
    "https://www.geeksforgeeks.org/how-to-add-header-row-to-a-pandas-dataframe/",
    "https://discuss.streamlit.io/t/is-it-possible-to-center-data-on-cell-like-excel/32045/7",
]

In [74]:
import pandas as pd
from matplotlib import pyplot
from seaborn.palettes import mpl_palette as colors
from sklearn import model_selection, neighbors
from typing import Literal

In [75]:
targetClass = "Outcome"
columnNames = [
    "Top-Left", "Top-Middle", "Top-Right",
    "Middle-Left", "Middle-Middle", "Middle-Right",
    "Bottom-Left", "Bottom-Middle", "Bottom-Right",
    targetClass,
]
dataset = pd.read_csv(
    "tic-tac-toe.data",
    names=columnNames,
)
dataLinesCount = 958

In [76]:
X_VALUE = 1
B_VALUE = 0
O_VALUE = -1

def cell_to_number(cell: Literal['b', 'x', 'o']):
  if cell == 'b':
    return B_VALUE

  if cell == 'x':
    return X_VALUE

  if cell == 'o':
    return O_VALUE

  return cell

In [77]:
def o_has_won(board: list[str]) -> bool:
    if len(board) != 9:
        raise ValueError("Board cannot have any different number of rows x columns than 9")

    for i in range(0, 9, 3):
        line = board[i:i+3]
        # print("Line: ", line)
        if all(cell == O_VALUE for cell in line):
            return True

    for i in range(0, 3):
        column = [board[i], board[i+3], board[i+6]]
        # print("Column: ", column)
        if all(cell == O_VALUE for cell in column):
            return True

    if board[4] != O_VALUE:
        return False

    diagonal1 = [board[0], board[8]]
    # print("D1: ", diagonal1)
    if all(cell == O_VALUE for cell in diagonal1):
        return True

    diagonal2 = [board[2], board[6]]
    # print("D2: ", diagonal2)
    if all(cell == O_VALUE for cell in diagonal2):
        return True

    return False

In [ ]:
dataset.head(5)

In [78]:
for columnName in columnNames:
  dataset[columnName] = dataset[columnName].apply(cell_to_number)

In [ ]:
dataset.head(5)

In [79]:
# NOTE: Run only once
for index, row in dataset.iterrows():
    board = row.array.tolist()[:-1]

    if row[targetClass] == 'positive':
        targetValue = "X Ganhou"
    elif o_has_won(board):
        targetValue = "O Ganhou"
    elif B_VALUE in board:
        targetValue = "Tem Jogo"
    else:
        targetValue = "Deu Velha"

    dataset.at[index, targetClass] = targetValue

In [ ]:
x_ganhou_count = dataset[targetClass].eq('X Ganhou').sum()
o_ganhou_count = dataset[targetClass].eq('O Ganhou').sum()
deu_velha_count = dataset[targetClass].eq('Deu Velha').sum()
tem_jogo_count = dataset[targetClass].eq('Tem Jogo').sum()

# Define the outcomes and their counts
outcomes = {
    'X Ganhou': x_ganhou_count,
    'O Ganhou': o_ganhou_count,
    'Deu Velha': deu_velha_count,
    'Tem Jogo': tem_jogo_count,
}

print(outcomes)

In [84]:
# Define the features (X) and the target variable (outcome)
X = dataset.drop(columns=[targetClass])
y = dataset[targetClass]

# Define the split ratios
split_ratios = {'train': 0.7, 'validation': 0.15, 'test': 0.15}
validation_plus_test_ratio = split_ratios['validation'] + split_ratios['test']
test_to_validation_ratio = split_ratios['test'] / validation_plus_test_ratio

# Split the data into training, validation, and test sets
X_train, X_temp, y_train, y_temp = model_selection.train_test_split(X, y, train_size=split_ratios['train'], stratify=y, random_state=42)
X_val, X_test, y_val, y_test = model_selection.train_test_split(X_temp, y_temp, test_size=test_to_validation_ratio, stratify=y_temp, random_state=42)

# Print the shapes of the resulting sets to verify the split
# Print the distribution of outcomes in each set
print("\nTraining set shape:", X_train.shape)
print("Training set distribution:")
print(y_train.value_counts(normalize=True))

print("\nValidation set shape:", X_val.shape)
print("Validation set distribution:")
print(y_val.value_counts(normalize=True))

print("\nTest set shape:", X_test.shape)
print("Test set distribution:")
print(y_test.value_counts(normalize=True))

{'X Ganhou': 626, 'O Ganhou': 316, 'Deu Velha': 16, 'Tem Jogo': 0}

Training set shape: (670, 9)
Training set distribution:
Outcome
X Ganhou     0.653731
O Ganhou     0.329851
Deu Velha    0.016418
Name: proportion, dtype: float64

Validation set shape: (144, 9)
Validation set distribution:
Outcome
X Ganhou     0.652778
O Ganhou     0.333333
Deu Velha    0.013889
Name: proportion, dtype: float64

Test set shape: (144, 9)
Test set distribution:
Outcome
X Ganhou     0.652778
O Ganhou     0.326389
Deu Velha    0.020833
Name: proportion, dtype: float64


In [ ]:
# @title Win for X
dataset.groupby('Win for X').size().plot(kind='barh', color=colors('Dark2'))
pyplot.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
train_set, test_set = model_selection.train_test_split(dataset, train_size=0.8)

kNN_classifier = neighbors.KNeighborsClassifier(n_neighbors=5)

In [ ]:
train_set_x = train_set.drop(columns=[targetClass])
train_set_y = train_set[targetClass]

kNN_classifier.fit(train_set_x, train_set_y)

In [ ]:
test_set_x = test_set.drop(columns=[targetClass])
test_set_y = test_set[targetClass].values

predictions = kNN_classifier.predict(test_set_x)

success_count = 0
print('\t', "Esperado | Obtido")
for i in range(0, len(test_set_y)):
  if i == len(predictions):
    break
  if equals := (predictions[i] == test_set_y[i]):
    success_count += 1
    continue
  print("\033[31m", f"Erro na predição da linha {i}:", "\033[0m")
  print('\t', predictions[i], '|', test_set_y[i], '\n')

error_count = len(predictions) - success_count
if error_count == 0:
  # Remove print "Esperado | Obtido"
  print('\b' * len("\tEsperado | Obtido"))

print("Predições Corretas", '|', "Erros")
print('\t      ', success_count, '|', error_count)